In [1]:
!pip install -U transformers datasets evaluate scikit-learn

In [2]:
from google.colab import drive
import pandas as pd

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Load the CSV
csv_path = '/content/drive/MyDrive/plant_training/text_corpus.csv'
df = pd.read_csv(csv_path)

print(df.head())
print("Total rows:", len(df))
print("Classes:", df.class_name.nunique())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
         class_name                                               text
0  Apple_Apple_Scab  Olive to dark-brown scab lesions on leaf surfa...
1  Apple_Apple_Scab  The plant leaf shows Olive to dark-brown scab ...
2  Apple_Apple_Scab  Farm inspection reveals Olive to dark-brown sc...
3  Apple_Apple_Scab  Visual diagnosis: Olive to dark-brown scab les...
4  Apple_Apple_Scab  The crop displays Olive to dark-brown scab les...
Total rows: 726
Classes: 58


In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["label"] = le.fit_transform(df["class_name"])

num_labels = df["label"].nunique()

print("num_labels:", num_labels)

num_labels: 58


In [4]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["label"],
    random_state=42
)

In [5]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df[["text","label"]])
val_dataset = Dataset.from_pandas(val_df[["text","label"]])

train_dataset = train_dataset.map(tokenize)
val_dataset = val_dataset.map(tokenize)

train_dataset = train_dataset.rename_column("label","labels")
val_dataset = val_dataset.rename_column("label","labels")

train_dataset.set_format("torch")
val_dataset.set_format("torch")

Map:   0%|          | 0/617 [00:00<?, ? examples/s]

Map:   0%|          | 0/109 [00:00<?, ? examples/s]

In [17]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification

model_name = "roberta-base"

tokenizer = RobertaTokenizer.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(
    model_name,
    num_labels=58
)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10,
    weight_decay=0.01
)

In [19]:
import numpy as np
from evaluate import load

metric = load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return metric.compute(predictions=preds, references=labels)

In [20]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [21]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,4.124961,4.046946,0.027523
2,3.574580,3.031680,0.614679
3,2.126798,1.672619,0.954128
4,1.209606,0.878268,0.990826
5,0.650229,0.472891,1.000000
6,0.359132,0.237493,1.000000
7,0.229331,0.133226,1.000000
8,0.134870,0.077603,1.000000
9,0.083056,0.048224,1.000000
10,0.058165,0.035321,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1170, training_loss=0.9374929831323461, metrics={'train_runtime': 497.964, 'train_samples_per_second': 18.586, 'train_steps_per_second': 2.35, 'total_flos': 609079296360960.0, 'train_loss': 0.9374929831323461, 'epoch': 15.0})

In [9]:
import transformers
print(transformers.__version__)

5.3.0


In [22]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

text = "tomato leaves show yellow mosaic patterns and curling edges"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

inputs = {k: v.to(device) for k, v in inputs.items()}   # move inputs to GPU

outputs = model(**inputs)

pred = outputs.logits.argmax().item()

print(le.inverse_transform([pred]))

['Corn_Maize_Northern_Leaf_Blight']


In [16]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

text = "grape leaves show black circular fungal lesions"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

pred = outputs.logits.argmax().item()

print("Predicted class:", le.inverse_transform([pred]))

Predicted class: ['Grape_Black_Rot']
